# 03. キャリブレーション — 量子化調整

**前提ファイル**: `network.onnx`, `settings.json`
（`01_onnx.ipynb`, `02_settings.ipynb` を実行済み）

In [1]:
try:
    import google.colab, subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ezkl", "onnx", "torch", "torchvision"])
except ImportError:
    pass

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import ezkl, json, torch, os

model_path    = '/content/drive/MyDrive/ezkl/network.onnx'
settings_path = '/content/drive/MyDrive/ezkl/settings.json'
cal_path      = '/content/drive/MyDrive/ezkl/calibration.json'

for f in [model_path, settings_path]:
    assert os.path.exists(f), f"{f} がありません。前のノートブックを先に実行してください。"

## なぜ量子化が必要か

ZK 回路（Groth16）は**有限体上の整数演算**しかできません。

PyTorch の浮動小数点 → 整数に変換する必要があります。

```
PyTorch: 0.314159...（float32）
    ↓ × 2^scale
ZK回路: 5193（整数）
```

scale が大きい → 精度は高いが回路が大きくなる（証明が遅い）  
scale が小さい → 回路は小さいが精度が落ちる

`calibrate_settings` は実データを使って**最適な scale を自動探索**します。

In [6]:
# キャリブレーション前の設定を確認
with open(settings_path) as f:
    before = json.load(f)
print(f"キャリブレーション前:")
print(f"  num_logrows     : {before.get('num_logrows', 'まだなし')}")
print(f"  input_scales    : {before.get('model_input_scales', 'まだなし')}")

キャリブレーション前:
  num_logrows     : まだなし
  input_scales    : [7]


In [9]:
before

{'run_args': {'input_scale': 7,
  'param_scale': 7,
  'rebase_scale': None,
  'scale_rebase_multiplier': 1,
  'lookup_range': [-32768, 32768],
  'logrows': 17,
  'num_inner_cols': 2,
  'variables': [['batch_size', 1]],
  'input_visibility': 'Public',
  'output_visibility': 'Public',
  'param_visibility': 'Fixed',
  'rebase_frac_zero_constants': False,
  'check_mode': 'UNSAFE',
  'decomp_base': 16384,
  'decomp_legs': 2,
  'bounded_log_lookup': False,
  'ignore_range_check_inputs_outputs': False,
  'epsilon': 2.220446049250313e-16,
  'disable_freivalds': False},
 'num_rows': 23396,
 'total_assignments': 46792,
 'total_const_size': 128,
 'total_dynamic_col_size': 0,
 'max_dynamic_input_len': 0,
 'num_dynamic_lookups': 0,
 'num_shuffles': 0,
 'total_shuffle_col_size': 0,
 'einsum_params': {'equations': [['abcde,abcde->abcd',
    {'c': 28, 'b': 1, 'a': 1, 'e': 2, 'd': 28}],
   ['abcde,abcde->abcd', {'e': 2, 'c': 12, 'a': 1, 'b': 2, 'd': 12}],
   ['abcde,abcde->abcd', {'e': 2, 'a': 1, 'c': 

In [7]:
# キャリブレーション用サンプルデータ（実データに近い分布で作る）
shape      = [1, 1, 28, 28]
data_array = (torch.rand(20, *shape[1:]) * 2 - 1).flatten().tolist()
json.dump({"input_data": [data_array]}, open(cal_path, 'w'))
print(f"サンプル数: {len(data_array)} 値")

サンプル数: 15680 値


In [ ]:
# "resources" = 回路サイズ最小化優先（証明時間・メモリを抑える）
# "accuracy"  = 精度優先（回路が大きくなる）
ezkl.calibrate_settings(cal_path, model_path, settings_path, "resources")
print("完了")

In [ ]:
with open(settings_path) as f:
    after = json.load(f)

logrows = after.get('num_logrows', 0)
print(f"キャリブレーション後:")
print(f"  num_logrows  : {logrows}")
print(f"  回路の行数   : 2^{logrows} = {2**logrows:,} 行")
print(f"  input_scales : {after.get('model_input_scales', 'N/A')}")

## `logrows` とは

`logrows = N` のとき回路の行数 = `2^N`。

```
logrows = 15 → 2^15 =  32,768 行（小さい・速い）
logrows = 17 → 2^17 = 131,072 行（標準）
logrows = 20 → 2^20 = 1,048,576 行（大きい・遅い）
```

`resources` モードは logrows を小さく抑えます。誤差レポート（Numerical Fidelity Report）が出ますが、`True` が返れば正常です。

---
次: `04_compile.ipynb` で ONNX を ZK 回路にコンパイルします。